# Colab T4 server
Select Runtime > Change runtime type > T4 GPU.

In [ ]:
%pip install -q "transformers>=4.45,<5" accelerate bitsandbytes fastapi uvicorn pyngrok

In [ ]:
import os, secrets, getpass, torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
from google.colab import files
print('Upload colab_server.py')
files.upload()
os.environ['AGENT_API_KEY'] = secrets.token_urlsafe(32)
from pyngrok import ngrok
ngrok.set_auth_token(getpass.getpass('ngrok authtoken: '))

In [ ]:
import subprocess, time, requests
log = open('server.log', 'w')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'colab_server:app', '--host', '0.0.0.0', '--port', '8000'], stdout=log, stderr=log)
for _ in range(360):
    if server.poll() is not None:
        raise RuntimeError(open('server.log').read())
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    server.terminate()
    raise TimeoutError('Inspect server.log')
print('Ngrok URL:', ngrok.connect(8000).public_url)
print('Server API key:', os.environ['AGENT_API_KEY'])
print('Keep runtime alive. Clear output before sharing.')